In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader
import torch

In [ ]:
batch_size = 4
max_new_tokens = 3

In [ ]:
import json

with open('baly_filtered_topic.json','r', encoding="utf8") as file:
    ori_data = json.load(file)

news_data = []
topic = 'abortion'
for nd in ori_data[topic]:
    if nd['bias_text'] != 'center':
        news_data.append(nd)

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
batch_size = 16

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)
model.eval()

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, raw_data, tokenizer):
        self.entries = []
        for sample in raw_data:
            user_input = sample['title'] + "\n" + sample['content_original']
            chat = [
                {"role": "system", 
                    "content": "You are a helpful assistant that classifies the political leaning of news articles as 'left', 'center', or 'right'."},
                {"role": "user", 
                    "content": f"Classify the political leaning of the following news article:\n\n{user_input}\n\nOnly respond with one of: left, center, or right."}
            ]
            prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
            self.entries.append({
                "id": sample["ID"],
                "bias": sample['bias_text'],
                "topic": sample["topic"],
                "original_text": user_input,
                "prompt": prompt
            })

        toks = tokenizer([e["prompt"] for e in self.entries], padding=False, truncation=True)
        lengths = [len(x) for x in toks["input_ids"]]
        sorted_idx = sorted(range(len(self.entries)), key=lambda i: lengths[i])
        self.entries = [self.entries[i] for i in sorted_idx]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        return self.entries[idx]

In [ ]:
def collate_fn(batch):
    prompts = [item["prompt"] for item in batch]
    encoded = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
    encoded = {k: v.to(model.device) for k, v in encoded.items()}
    metadata = [{"id": item["id"], "topic": item["topic"], "bias": item["bias"], "original_text": item["original_text"]} for item in batch]
    return encoded, metadata

In [ ]:
dataset = NewsDataset(news_data, tokenizer)
dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_fn)

In [ ]:
results = []
with torch.no_grad():
    for inputs, metadata in dataloader:
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        for meta, text in zip(metadata, decoded):
            results.append({
                "id": meta["id"],
                "topic": meta["topic"],
                "bias": meta["bias"],
                "original_text": meta["original_text"],
                "response": text
            })

In [ ]:
import requests
import time
from tqdm import tqdm
from collections import deque

url = ""
MODEL = ""

In [ ]:
results = []

for sample in tqdm(news_data):
    
    user_input = sample['title'] + "\n" + sample['content_original']
    
    payload = {
        "model": MODEL,
        "stream": False,
        "max_tokens": 3,
        "messages": [
                    {"role": "system", 
                        "content": "You are a helpful assistant that classifies the political leaning of news articles as 'left', 'center', or 'right'."},
                    {"role": "user", 
                        "content": f"Classify the political leaning of the following news article:\n\n{user_input}\n\nOnly respond with one of: left, center, or right."}
            ]
    }
    headers = {
        "Authorization": "",
        "Content-Type": "application/json"
    }

    response = requests.request("POST", url, json=payload, headers=headers).json()
    while 'choices' not in response:
        time.sleep(30)
        response = requests.request("POST", url, json=payload, headers=headers).json()
    pred = response['choices'][0]['message']['content']
    results.append({
                "id": sample["ID"],
                "topic": sample["topic"],
                "bias": sample['bias_text'],
                "response": pred
            })


In [ ]:
with open("stance_prediction_abortion_"+MODEL+".json", "w") as json_file:
  json.dump(results, json_file, indent=4)

# Analyze Rsults

In [ ]:
import json

with open('stance_prediction_abortion_qwen.json','r', encoding="utf8") as file:
    results = json.load(file)

In [ ]:
statistics = {'left_as_center': 0,
                'right_as_center': 0,
                'left_as_right': 0,
                'right_as_left': 0,
                'left_as_left': 0,
                'right_as_right': 0
                }
for result in results:
    if result['response'].lower() not in ['left', 'right', 'center']:
        continue
    if result['bias'] == 'left' and result['response'].lower() == 'center': statistics['left_as_center'] += 1
    if result['bias'] == 'right' and result['response'].lower() == 'center': statistics['right_as_center'] += 1
    if result['bias'] == 'left' and result['response'].lower() == 'right': statistics['left_as_right'] += 1
    if result['bias'] == 'right' and result['response'].lower() == 'left': statistics['right_as_left'] += 1
    if result['bias'] == 'left' and result['response'].lower() == 'left': statistics['left_as_left'] += 1
    if result['bias'] == 'right' and result['response'].lower() == 'right': statistics['right_as_right'] += 1

In [ ]:
with open("stance_qwen_statistics.json", "w") as json_file:
  json.dump(statistics, json_file, indent=4)

# Plot Results

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter


def plot_left_bar(count_A, count_B, count_C):
    counts = [count_A, count_B, count_C]
    labels = ['left', 'center', 'right']
    colors = ['blue', 'green', 'red']

    # Plot
    fig, ax = plt.subplots(figsize=(6, 0.2))  
    ax.barh(0, counts[0], color=colors[0], edgecolor='white', label=labels[0])
    ax.barh(0, counts[1], left=counts[0], color=colors[1], edgecolor='white', label=labels[1])
    ax.barh(0, counts[2], left=counts[0]+counts[1], color=colors[2], edgecolor='white', label=labels[2])

    # Remove axes
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_xlim(0, sum(counts))
    plt.box(False)
    plt.show()
    
def plot_right_bar(count_A, count_B, count_C):
    counts = [count_C, count_B, count_A]
    labels = ['right', 'center', 'left']
    colors = ['red', 'green', 'blue']

    # Plot
    fig, ax = plt.subplots(figsize=(6, 0.2))  
    ax.barh(0, counts[0], color=colors[0], edgecolor='white', label=labels[0])
    ax.barh(0, counts[1], left=counts[0], color=colors[1], edgecolor='white', label=labels[1])
    ax.barh(0, counts[2], left=counts[0]+counts[1], color=colors[2], edgecolor='white', label=labels[2])

    # Remove axes
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_xlim(0, sum(counts))
    plt.box(False)
    plt.show()

In [ ]:
import json

with open('stance_qwen_statistics.json','r', encoding="utf8") as file:
    results = json.load(file)

In [ ]:
race = results["race"]
immigration = results["immigration"]
election = results["election"]
lgbt = results["lgbt"]
abortion = results["abortion"]

In [ ]:
plot_left_bar(abortion["left_as_left"],abortion["left_as_center"], abortion["left_as_right"])
plot_right_bar(abortion["right_as_left"],abortion["right_as_center"], abortion["right_as_right"])

In [ ]:
plot_left_bar(race["left_as_left"],race["left_as_center"], race["left_as_right"])
plot_right_bar(race["right_as_left"],race["right_as_center"], race["right_as_right"])

In [ ]:
plot_left_bar(immigration["left_as_left"],immigration["left_as_center"], immigration["left_as_right"])
plot_right_bar(immigration["right_as_left"],immigration["right_as_center"], immigration["right_as_right"])

In [ ]:
plot_left_bar(election["left_as_left"],election["left_as_center"], election["left_as_right"])
plot_right_bar(election["right_as_left"],election["right_as_center"], election["right_as_right"])